# BookMatch — прототип рекомендаційної системи книг
**Етап 2 міні-проєкту: прототип системи**

Датасет: [Book Recommendation Dataset (Book-Crossing)](https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset) — `Books.csv`, `Users.csv`, `Ratings.csv`.

**План прототипу**
1. Завантаження та аналіз даних (EDA)
2. Очищення та попередня обробка
3. Розбиття на навчальну та тестову вибірки
4. **Модель 1** — item-based kNN (колаборативна фільтрація, косинусна подібність)
5. **Модель 2** — матрична факторизація (SVD з базовими зміщеннями)
6. Порівняння моделей за метриками RMSE, MAE, Precision@K, Recall@K
7. Демонстрація рекомендацій і висновки

## 0. Імпорт бібліотек і налаштування

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Параметри прототипу (можна змінювати)
MIN_BOOK_RATINGS = 10   # залишаємо книги, які мають щонайменше стільки явних оцінок
MIN_USER_RATINGS = 5    # і користувачів, які поставили щонайменше стільки оцінок
TEST_SIZE = 0.2         # частка оцінок кожного користувача, що йде в тест
K_NEIGHBORS = 30        # кількість сусідів у kNN
TOP_K = 10              # довжина списку рекомендацій для Precision@K / Recall@K
RELEVANT = 8            # оцінка ≥ 8 вважається «книга сподобалась»

## 1. Завантаження даних
Покладіть три CSV-файли в папку `data/` поруч із ноутбуком **або** запустіть комірку нижче в Google Colab — вона спробує завантажити датасет через `kagglehub`.

In [ ]:
DATA_DIR = "data"

if not os.path.exists(os.path.join(DATA_DIR, "Books.csv")):
    try:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download("arashnic/book-recommendation-dataset")
    except Exception as e:
        print("Не вдалося завантажити через kagglehub:", e)
        print("Завантажте датасет вручну з Kaggle і покладіть CSV у папку data/")
print("Дані беремо з:", DATA_DIR)

def read_csv(name):
    path = os.path.join(DATA_DIR, name)
    try:
        return pd.read_csv(path, low_memory=False, dtype={"ISBN": str})
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, dtype={"ISBN": str}, encoding="latin-1")

books = read_csv("Books.csv")
users = read_csv("Users.csv")
ratings = read_csv("Ratings.csv")

for name, df in [("Books", books), ("Users", users), ("Ratings", ratings)]:
    print(f"{name:8s}: {df.shape[0]:>9,} рядків, {df.shape[1]} колонок")

In [ ]:
books.head(3)

In [ ]:
users.head(3)

In [ ]:
ratings.head(3)

## 2. Аналіз даних (EDA)

### 2.1. Пропуски

In [ ]:
for name, df in [("Books", books), ("Users", users), ("Ratings", ratings)]:
    miss = df.isna().mean().mul(100).round(1)
    print(f"--- {name} (% пропусків) ---")
    print(miss[miss > 0].to_string() if (miss > 0).any() else "пропусків немає")

### 2.2. Розподіл оцінок
Оцінка **0** у Book-Crossing — це *неявна* взаємодія (користувач мав справу з книгою, але не оцінив її), а не «погана» оцінка. Тому явні (1–10) та неявні (0) оцінки розглядаємо окремо.

In [ ]:
implicit_share = (ratings["Book-Rating"] == 0).mean()
print(f"Частка неявних оцінок (0): {implicit_share:.1%}")

explicit = ratings[ratings["Book-Rating"] > 0]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ratings["Book-Rating"].value_counts().sort_index().plot.bar(ax=ax[0], color="#888")
ax[0].set_title("Усі оцінки (0 = неявна)"); ax[0].set_xlabel("Оцінка"); ax[0].set_ylabel("Кількість")
explicit["Book-Rating"].value_counts().sort_index().plot.bar(ax=ax[1], color="#3a6ea5")
ax[1].set_title("Явні оцінки 1–10"); ax[1].set_xlabel("Оцінка")
plt.tight_layout(); plt.show()
print(f"Середня явна оцінка: {explicit['Book-Rating'].mean():.2f}")

### 2.3. Активність користувачів і популярність книг (розрідженість)

In [ ]:
per_user = explicit.groupby("User-ID").size()
per_book = explicit.groupby("ISBN").size()
print(f"Користувачів з 1 явною оцінкою: {(per_user == 1).mean():.1%}")
print(f"Книг з 1 явною оцінкою:          {(per_book == 1).mean():.1%}")
density = len(explicit) / (per_user.size * per_book.size)
print(f"Щільність матриці «користувач × книга»: {density:.6%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
per_user.clip(upper=50).plot.hist(bins=50, ax=ax[0], color="#3a6ea5")
ax[0].set_title("Оцінок на користувача (обрізано на 50)"); ax[0].set_yscale("log")
per_book.clip(upper=50).plot.hist(bins=50, ax=ax[1], color="#c0504d")
ax[1].set_title("Оцінок на книгу (обрізано на 50)"); ax[1].set_yscale("log")
plt.tight_layout(); plt.show()

### 2.4. Вік і країни користувачів

In [ ]:
age = pd.to_numeric(users["Age"], errors="coerce")
print(f"Пропуски у віці: {age.isna().mean():.1%};  нереалістичний вік (<5 або >100): {((age < 5) | (age > 100)).mean():.1%}")
country = users["Location"].astype(str).str.split(",").str[-1].str.strip().replace("", np.nan)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
age[(age >= 5) & (age <= 100)].plot.hist(bins=40, ax=ax[0], color="#3a6ea5")
ax[0].set_title("Вік користувачів (5–100)")
country.value_counts().head(10).plot.barh(ax=ax[1], color="#9bbb59")
ax[1].set_title("Топ-10 країн"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 3. Очищення та попередня обробка

In [ ]:
# --- Книги ---
books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
bad_year = (books["Year-Of-Publication"] < 1800) | (books["Year-Of-Publication"] > 2006)
books.loc[bad_year, "Year-Of-Publication"] = np.nan   # дані зібрано у 2004 р.
books["ISBN"] = books["ISBN"].str.strip().str.upper()
ratings["ISBN"] = ratings["ISBN"].astype(str).str.strip().str.upper()
books = books.drop_duplicates("ISBN")
books["Book-Author"] = books["Book-Author"].fillna("Unknown")
books["Publisher"] = books["Publisher"].fillna("Unknown")

# --- Користувачі ---
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
users.loc[(users["Age"] < 5) | (users["Age"] > 100), "Age"] = np.nan
users["Country"] = users["Location"].astype(str).str.split(",").str[-1].str.strip()

# --- Оцінки: тільки явні та тільки для книг, що є в каталозі ---
before = len(ratings)
r = ratings[(ratings["Book-Rating"] > 0) & ratings["ISBN"].isin(books["ISBN"])].copy()
print(f"Оцінок було: {before:,};  явних і з відомими книгами: {len(r):,}")

# --- Фільтр розрідженості: популярні книги й активні користувачі ---
book_cnt = r["ISBN"].value_counts()
r = r[r["ISBN"].isin(book_cnt[book_cnt >= MIN_BOOK_RATINGS].index)]
user_cnt = r["User-ID"].value_counts()
r = r[r["User-ID"].isin(user_cnt[user_cnt >= MIN_USER_RATINGS].index)]

n_users, n_books = r["User-ID"].nunique(), r["ISBN"].nunique()
assert n_users > 0 and n_books > 0, "Після фільтрації нічого не лишилось — зменшіть MIN_BOOK_RATINGS / MIN_USER_RATINGS"
print(f"Після фільтрації: {len(r):,} оцінок, {n_users:,} користувачів, {n_books:,} книг")
print(f"Щільність матриці: {len(r) / (n_users * n_books):.4%}")

## 4. Розбиття на навчальну та тестову вибірки
Для кожного користувача випадково відкладаємо 20% його оцінок у тест. Так кожен тестовий користувач має історію в навчальній вибірці, і модель може дати йому рекомендації.

In [ ]:
# Кодуємо користувачів і книги індексами 0..n-1
user_ids = np.sort(r["User-ID"].unique()); book_ids = np.sort(r["ISBN"].unique())
u_idx = {u: i for i, u in enumerate(user_ids)}; b_idx = {b: i for i, b in enumerate(book_ids)}
r["u"] = r["User-ID"].map(u_idx); r["i"] = r["ISBN"].map(b_idx)

test = r.groupby("u", group_keys=False).sample(frac=TEST_SIZE, random_state=RANDOM_STATE)
train = r.drop(test.index)
# книги, яких немає у train, неможливо передбачити жодною моделлю
test = test[test["i"].isin(train["i"].unique())]
print(f"Train: {len(train):,} оцінок | Test: {len(test):,} оцінок")

n_u, n_i = len(user_ids), len(book_ids)
R = csr_matrix((train["Book-Rating"].astype(float), (train["u"], train["i"])), shape=(n_u, n_i))

y_test = test["Book-Rating"].values.astype(float)
tu, ti = test["u"].values, test["i"].values
def rmse(y, p): return float(np.sqrt(np.mean((y - p) ** 2)))
def mae(y, p):  return float(np.mean(np.abs(y - p)))

## 5. Базова лінія (для довідки)
Простий прогноз: **глобальне середнє + зміщення користувача + зміщення книги** (з регуляризацією). Обидві моделі мають бути кращими за неї — інакше вони не мають сенсу. Ці ж зміщення використовує модель SVD.

In [ ]:
def fit_biases(train, reg=10.0, n_iter=10):
    mu = train["Book-Rating"].mean()
    bu, bi = np.zeros(n_u), np.zeros(n_i)
    u, i, y = train["u"].values, train["i"].values, train["Book-Rating"].values.astype(float)
    for _ in range(n_iter):  # почергове оновлення (ALS для зміщень)
        bi = np.bincount(i, y - mu - bu[u], n_i) / (np.bincount(i, minlength=n_i) + reg)
        bu = np.bincount(u, y - mu - bi[i], n_u) / (np.bincount(u, minlength=n_u) + reg)
    return mu, bu, bi

mu, bu, bi = fit_biases(train)
predict_baseline = lambda u, i: np.clip(mu + bu[u] + bi[i], 1, 10)
print(f"Глобальне середнє: {mu:.2f}")

## 6. Модель 1 — item-based kNN (колаборативна фільтрація)
**Ідея:** книга `i` сподобається користувачу, якщо йому сподобались *схожі* книги. Дві книги схожі, якщо їх однаково оцінюють ті самі люди.

- **Подібність:** скоригована косинусна міра (adjusted cosine) між стовпцями матриці оцінок, з яких віднято середню оцінку кожного користувача (лекція 4 — метричні методи).
- **Прогноз:**
$$\hat r_{ui} = \bar r_u + \frac{\sum_{j \in N_k(i;u)} s_{ij}\,(r_{uj} - \bar r_u)}{\sum_{j \in N_k(i;u)} |s_{ij}|}$$
де $N_k(i;u)$ — до $k$ найсхожіших на `i` книг серед тих, що оцінив користувач `u`.

In [ ]:
class ItemKNN:
    def __init__(self, k=30, shrink=10):
        self.k, self.shrink = k, shrink

    def fit(self, R):
        self.R = R.tocsr()
        counts = np.diff(self.R.indptr)
        self.user_mean = np.asarray(self.R.sum(1)).ravel() / np.maximum(counts, 1)
        Rc = self.R.copy().astype(float)
        Rc.data -= np.repeat(self.user_mean, counts)        # центрування по користувачу
        self.Rc = Rc.tocsr()
        self.Xn = normalize(Rc.tocsc(), axis=0).tocsc()      # нормовані стовпці-книги
        return self

    def _predict_user(self, u, items):
        start, end = self.R.indptr[u], self.R.indptr[u + 1]
        rated = self.R.indices[start:end]
        dev = self.Rc.data[start:end]                        # r_uj - mean_u
        if len(rated) == 0:
            return np.full(len(items), np.nan)
        S = (self.Xn[:, items].T @ self.Xn[:, rated]).toarray()   # |items| × |rated|
        if S.shape[1] > self.k:                              # лишаємо k найсхожіших
            thr = -np.partition(-S, self.k - 1, axis=1)[:, self.k - 1:self.k]
            S = np.where(S >= thr, S, 0.0)
        S = np.maximum(S, 0.0)                               # беремо лише позитивну подібність
        num, den = S @ dev, S.sum(1)
        pred = self.user_mean[u] + np.divide(num, den, out=np.zeros_like(num), where=den > 0)
        return np.clip(pred, 1, 10)

    def predict(self, users, items):
        out = np.empty(len(users))
        df = pd.DataFrame({"u": users, "i": items, "pos": np.arange(len(users))})
        for u, g in df.groupby("u"):
            out[g["pos"].values] = self._predict_user(u, g["i"].values)
        return out

    def similar_items(self, i, n=10):
        s = (self.Xn[:, [i]].T @ self.Xn).toarray().ravel()
        s[i] = -np.inf
        top = np.argpartition(-s, n)[:n]
        return top[np.argsort(-s[top])], np.sort(s[top])[::-1]

t0 = time.time()
knn = ItemKNN(k=K_NEIGHBORS).fit(R)
print(f"kNN навчено за {time.time() - t0:.1f} с")

## 7. Модель 2 — матрична факторизація (SVD)
**Ідея:** кожного користувача та кожну книгу описуємо вектором із $f$ прихованих факторів (умовно — «жанр», «складність», «сучасність»). Оцінка — це скалярний добуток цих векторів плюс зміщення:
$$\hat r_{ui} = \mu + b_u + b_i + p_u^\top q_i$$
Вектори знаходимо усіченим сингулярним розкладом (truncated SVD) матриці залишків $r_{ui} - (\mu + b_u + b_i)$. Це метод зниження розмірності (лекція 6), застосований до матриці оцінок.

In [ ]:
class BiasedSVD:
    def __init__(self, n_factors=50):
        self.f = n_factors

    def fit(self, train, R_shape):
        self.mu, self.bu, self.bi = fit_biases(train)
        u, i = train["u"].values, train["i"].values
        resid = train["Book-Rating"].values - (self.mu + self.bu[u] + self.bi[i])
        Res = csr_matrix((resid, (u, i)), shape=R_shape)
        f = min(self.f, min(R_shape) - 1)
        U, s, Vt = svds(Res, k=f, random_state=RANDOM_STATE)
        self.P, self.Q = U * np.sqrt(s), (Vt.T * np.sqrt(s))
        return self

    def predict(self, users, items):
        dot = np.einsum("ij,ij->i", self.P[users], self.Q[items])
        return np.clip(self.mu + self.bu[users] + self.bi[items] + dot, 1, 10)

    def score_all(self, u):
        return self.mu + self.bu[u] + self.bi + self.Q @ self.P[u]

### 7.1. Підбір кількості факторів (на валідаційній вибірці)
Кількість факторів $f$ — гіперпараметр. Підбираємо його на **валідаційній** частині навчальних даних (20% оцінок кожного користувача з train), щоб не «підглядати» в тест. Забагато факторів — модель запам'ятовує шум (перенавчання).

In [ ]:
val = train.groupby("u", group_keys=False).sample(frac=0.2, random_state=RANDOM_STATE)
sub_train = train.drop(val.index)
val = val[val["i"].isin(sub_train["i"].unique())]
y_val, vu, vi = val["Book-Rating"].values.astype(float), val["u"].values, val["i"].values

factor_grid = [5, 10, 20, 50, 100]
grid_rmse = [rmse(y_val, BiasedSVD(f).fit(sub_train, (n_u, n_i)).predict(vu, vi)) for f in factor_grid]
best_f = factor_grid[int(np.argmin(grid_rmse))]

plt.figure(figsize=(6, 3.5))
plt.plot(factor_grid, grid_rmse, "o-", color="#c0504d")
plt.axvline(best_f, ls="--", color="#666")
plt.xlabel("Кількість факторів f"); plt.ylabel("RMSE на валідації"); plt.title("SVD: вплив кількості факторів")
plt.grid(alpha=.3); plt.show()
print("RMSE на валідації:", dict(zip(factor_grid, np.round(grid_rmse, 4))), "→ обрано f =", best_f)

# Фінальна модель навчається на всьому train з найкращим f
t0 = time.time()
svd = BiasedSVD(n_factors=best_f).fit(train, (n_u, n_i))
print(f"SVD (f={best_f}) навчено за {time.time() - t0:.1f} с")

## 8. Порівняння моделей
Дві групи метрик:
- **точність прогнозу оцінки** — RMSE та MAE (лекції 2–3, регресійні метрики); чим менше, тим краще;
- **якість списку рекомендацій** — Precision@K та Recall@K: скільки з топ-K рекомендованих книг користувач справді оцінив на ≥ 8 у тестовій вибірці; чим більше, тим краще.

Precision@K рахуємо на випадковій підвибірці користувачів, бо для кожного треба оцінити всі книги каталогу.

In [ ]:
t0 = time.time(); p_knn = knn.predict(tu, ti); t_knn = time.time() - t0
t0 = time.time(); p_svd = svd.predict(tu, ti); t_svd = time.time() - t0
p_base = predict_baseline(tu, ti)

In [ ]:
def precision_recall_at_k(score_fn, n_users_eval=500, k=TOP_K):
    rel = test[test["Book-Rating"] >= RELEVANT].groupby("u")["i"].apply(set)
    rng = np.random.default_rng(RANDOM_STATE)
    eval_users = rng.choice(rel.index.values, size=min(n_users_eval, len(rel)), replace=False)
    P, Rc = [], []
    for u in eval_users:
        scores = score_fn(u).astype(float)
        seen = R.indices[R.indptr[u]:R.indptr[u + 1]]
        scores[seen] = -np.inf                       # не рекомендуємо вже прочитане
        top = np.argpartition(-scores, k)[:k]
        hits = len(set(top) & rel[u])
        P.append(hits / k); Rc.append(hits / len(rel[u]))
    return float(np.mean(P)), float(np.mean(Rc))

all_items = np.arange(n_i)
score_knn  = lambda u: knn._predict_user(u, all_items)
score_svd  = lambda u: svd.score_all(u)
score_base = lambda u: mu + bu[u] + bi              # фактично «популярні книги»

pr = {name: precision_recall_at_k(fn) for name, fn in
      [("Базова лінія", score_base), ("Item-kNN", score_knn), ("SVD", score_svd)]}

In [ ]:
results = pd.DataFrame({
    "RMSE ↓": [rmse(y_test, p_base), rmse(y_test, p_knn), rmse(y_test, p_svd)],
    "MAE ↓":  [mae(y_test, p_base),  mae(y_test, p_knn),  mae(y_test, p_svd)],
    f"Precision@{TOP_K} ↑": [pr[m][0] for m in pr],
    f"Recall@{TOP_K} ↑":    [pr[m][1] for m in pr],
    "Час прогнозу, с": [np.nan, t_knn, t_svd],
}, index=["Базова лінія", "Item-kNN", "SVD"]).round(4)
results

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#999", "#3a6ea5", "#c0504d"]
for a, col in zip(ax, ["RMSE ↓", "MAE ↓", f"Precision@{TOP_K} ↑"]):
    results[col].plot.bar(ax=a, color=colors, rot=0)
    a.set_title(col)
    for p in a.patches:
        a.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                   ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Де модель помиляється сильніше: залежно від активності користувача
hist = np.diff(R.indptr)[tu]
bins = pd.cut(hist, [0, 5, 10, 20, 50, np.inf], labels=["≤5", "6–10", "11–20", "21–50", ">50"])
err = pd.DataFrame({"група": bins, "kNN": (y_test - p_knn) ** 2, "SVD": (y_test - p_svd) ** 2})
by_group = err.groupby("група", observed=True)[["kNN", "SVD"]].mean().pow(0.5)
by_group.plot.bar(color=["#3a6ea5", "#c0504d"], rot=0, figsize=(7, 3.5),
                  title="RMSE залежно від кількості оцінок користувача у train")
plt.ylabel("RMSE"); plt.show()

## 9. Демонстрація рекомендацій

In [ ]:
meta = books.set_index("ISBN")[["Book-Title", "Book-Author", "Year-Of-Publication"]]
def show(item_indices, extra=None, extra_name="score"):
    df = meta.loc[book_ids[item_indices]].reset_index()
    if extra is not None:
        df[extra_name] = np.round(extra, 3)
    return df

# Сценарій 1: «Схожі книги» (Модель 1, kNN) — для найпопулярнішої книги
popular = np.asarray((R > 0).sum(0)).ravel().argmax()
print("Обрана книга:", meta.loc[book_ids[popular], "Book-Title"])
idx, sims = knn.similar_items(popular, n=TOP_K)
show(idx, sims, "подібність")

In [ ]:
# Сценарій 2: персональні рекомендації (Модель 2, SVD) для випадкового активного користувача
u = int(np.argmax(np.diff(R.indptr)))
seen = R.indices[R.indptr[u]:R.indptr[u + 1]]
print(f"Користувач {user_ids[u]} — оцінив {len(seen)} книг. Його улюблені:")
top_seen = seen[np.argsort(-R.data[R.indptr[u]:R.indptr[u + 1]])][:5]
display(show(top_seen))

s = svd.score_all(u); s[seen] = -np.inf
rec = np.argsort(-s)[:TOP_K]
print("Рекомендації SVD:")
show(rec, np.clip(s[rec], 1, 10), "прогноз оцінки")

## 10. Висновки
Після запуску на повному датасеті заповніть числа з таблиці порівняння (розділ 8). Комірка нижче формує короткий підсумок автоматично.

In [ ]:
best_rmse = results.loc[["Item-kNN", "SVD"], "RMSE ↓"].idxmin()
best_prec = results.loc[["Item-kNN", "SVD"], f"Precision@{TOP_K} ↑"].idxmax()
print(f"• Дані після очищення: {len(r):,} явних оцінок, {n_u:,} користувачів, {n_i:,} книг.")
print(f"• Найточніший прогноз оцінки (RMSE): {best_rmse} — {results.loc[best_rmse, 'RMSE ↓']:.3f} "
      f"(базова лінія {results.loc['Базова лінія', 'RMSE ↓']:.3f}).")
print(f"• Найкращий список рекомендацій (Precision@{TOP_K}): {best_prec} — "
      f"{results.loc[best_prec, f'Precision@{TOP_K} ↑']:.4f}.")
print(f"• Оптимальна кількість факторів SVD (за валідацією): {best_f}.")

**Що варто сказати на захисті**
- **Чому дві моделі.** kNN — інтерпретований метод («рекомендуємо, бо вам сподобалась книга X»), добре підходить для сценарію «схожі книги». SVD стискає розріджену матрицю до кількох прихованих факторів, тому краще узагальнює, коли історія користувача коротка.
- **Чому такі метрики.** RMSE/MAE показують точність прогнозу оцінки, але користувач бачить не оцінку, а *список*, тому додано Precision@K і Recall@K. Модель із кращим RMSE не обов'язково дає кращий список.
- **Чому Precision@K маленький.** Матриця дуже розріджена: у тесті в користувача лише кілька оцінених книг серед тисяч, тому навіть влучання 1 з 10 — хороший результат. Важливо порівнювати моделі між собою та з базовою лінією.
- **Обмеження.** Дані 2004 року; неявні оцінки (0) поки не використано; нові користувачі без історії отримують лише популярні книги (проблема холодного старту).
- **Наступний етап (MVP).** Веб-інтерфейс на Streamlit: пошук книги → «схожі книги» (kNN) та персональні рекомендації (SVD); врахування неявних оцінок і контентних ознак (автор, видавництво) для холодного старту.